In [96]:
#!pip install yfinance

In [97]:
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from typing import Callable, Dict, List, Tuple
import yfinance as yf

In [98]:
def load_price_data(path: str) -> pd.DataFrame:
    """
    Load price data from CSV with at least:
        - Date
        - Close price
    """

    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    df = df[['Close', 'High', 'Low', 'Open', 'Volume']].astype(float)
    return df

In [99]:
def strategy_buy_and_hold(df: pd.DataFrame) -> pd.Series:
    """
    Always long (after first day)
    """

    signal = pd.Series(1, index=df.index)
    signal.iloc[0] = 0 # start flat on the first bar
    return signal

In [100]:
def strategy_sma_long_only(df: pd.DataFrame,
                           fast_window: int = 20,
                           slow_window: int = 50) -> pd.Series:
    """
    Long-only SMA crossover:
        - Long when fast SMA > slow SMA
        - Flat otherwise
    """

    tmp = df.copy()
    # Corrected 'long_window' to 'fast_window'
    tmp['SMA_fast'] = tmp['Close'].rolling(window=fast_window).mean()
    tmp['SMA_slow'] = tmp['Close'].rolling(window=slow_window).mean()

    signal = (tmp['SMA_fast'] > tmp['SMA_slow']).astype(int)

    # Shift by 1 bar to avoid lookahead bias
    signal = signal.shift(1).fillna(0)
    return signal

In [101]:
def strategy_sma_long_short(df: pd.DataFrame,
                            fast_window: int = 20,
                            slow_window: int = 50) -> pd.Series:
    """
    Long/short SMA crossover:
        - Long (1) when fast SMA > slow SMA
        - Short (-1) when fast SMA < slow SMA
        - Flat (0) when equal or not enough data
    """

    tmp = df.copy()
    tmp['SMA_fast'] = tmp['Close'].rolling(window=fast_window).mean()
    tmp['SMA_slow'] = tmp['Close'].rolling(window=slow_window).mean()

    signal = pd.Series(0, index=df.index)
    signal[(tmp['SMA_fast'] > tmp['SMA_slow'])] = 1
    signal[(tmp['SMA_fast'] < tmp['SMA_slow'])] = -1

    # Shift by 1 bar to avoid lookahead bias
    signal = signal.shift(1).fillna(0)
    return signal


In [102]:
def strategy_mean_reversion_long_short(df: pd.DataFrame,
                                       lookback: int = 5,
                                       z_entry: float = 1.0) -> pd.Series:
    """
    Simple mean-reversion on daily returns with long/short:
        - If today's return is more than z_entry std BELOW rolling mean -> long next bar
        - If today's return is more than z_entry std ABOVE rolling mean -> short next bar
        - Else flat
    """

    tmp = df.copy()
    tmp['ret'] = tmp['Close'].pct_change()

    mu = tmp['ret'].rolling(lookback).mean()
    sigma = tmp['ret'].rolling(lookback).std()
    z = (tmp['ret'] - mu) / sigma

    signal = pd.Series(0, index=tmp.index)
    signal[(z < -z_entry)] = 1  # big down move -> long
    signal[(z > z_entry)] = -1  # big up move -> short

    signal = signal.shift(1).fillna(0)
    return signal

In [103]:
def backtest(price_df: pd.DataFrame,
             signal: pd.Series, # Added signal as an explicit argument
             initial_capital: float = 10_0000.0,
             trading_fee_bps: float = 10.0) -> pd.DataFrame:
    """
    Generic backtest for a given signal.
    signal: Series aligned to price_df.index, values in {-1, 0, 1}
        1 = long, -1 = short, 0 = no position

    Assumes we put 100% of equity into the position (no leverage)

    Args:
        price_df (pd.DataFrame): DataFrame containing 'Close' prices.
        signal (pd.Series): Series aligned to price_df.index, values in {-1, 0, 1}.
        initial_capital (float): Starting cash for the backtest.
        trading_fee_bps (float): Transaction cost per trade as a percentage
          (e.g., 0.001 for 0.1%).

    Returns:
        pd.DataFrame: DataFrame with daily returns, strategy returns,
        and cumulative returns.
    """

    df = price_df.copy()

    # Align signal and ensure float
    df['signal'] = signal.astype(float)


    # Calculate daily percentage change in Close price
    df['asset_ret'] = df['Close'].pct_change().fillna(0.0)

    # Postition is the signal (0 or 1)
    # Note: strategy functions already shift signal by 1 for lookahead bias
    df['position'] = df['signal']

    # When position changes, we "trade"
    df['position_change'] = df['position'].diff().fillna(df['position'])

    fee_rate = trading_fee_bps / 10_0000.0  # bps -> decimal

    equity = initial_capital
    equity_series = []

    for i, row in df.iterrows():
        pos = row['position']
        asset_ret = row['asset_ret']

        # PnL: long/short exposure * return - THIS WAS THE MISSING CALCULATION!
        # long (1): equity *= (1 + r)
        # short (-1): equity *= (1 - r)
        equity *= (1 + pos * asset_ret)

        # Trading cost when position changes (including flip long <-> short)
        if row['position_change'] != 0:
            # abs(postition_change) = 1 for open/close, 2 for flip long <-> short
            # Cost applied to current equity
            traded_notional = abs(row['position_change']) * equity
            cost = traded_notional * fee_rate
            equity -= cost

        equity_series.append(equity)

    df['equity'] = equity_series
    df['strategy_ret'] = df['equity'].pct_change().fillna(0.0)
    df['cum_ret'] = (1 + df['strategy_ret']).cumprod()

    return df

In [104]:
def add_signals(df: pd.DataFrame,
                fast_window: int = 20,
                slow_window: int = 50) -> pd.DataFrame:
    """
    Add moving average signals.
    Signal is:
        1 -> long
        0 -> flat
        -1 -> short
    """

    df = df.copy()
    df['fast_ma'] = df['Close'].rolling(fast_window).mean()
    df['slow_ma'] = df['Close'].rolling(slow_window).mean()

    # Basic rule: long when fast > slow, else flat
    df['signal'] = 0.0
    df.loc[df['fast_ma'] > df['slow_ma'], 'signal'] = 1.0
    df.loc[df['fast_ma'] < df['slow_ma'], 'signal'] = -1.0

    # Shift signal by 1 bar to avoid lookahead bias
    df['signal'] = df['signal'].shift(1).fillna(0)
    return df

In [105]:
def performance_summary(df: pd.DataFrame,
                        risk_free_rate: float = 0.0) -> Dict[str, float]:
    """
    Basic performance stats from an equity curve.
    """

    trading_days = 252
    rets = df["strategy_ret"]

    cum_return = (1 + rets).prod() - 1

    if len(rets) > 0:
        # Corrected: Use the already calculated 'cum_return' for annualization
        ann_return = (1 + cum_return) ** (trading_days / len(rets)) - 1
    else:
        ann_return = np.nan

    ann_vol = rets.std() * np.sqrt(trading_days)

    if ann_vol > 0:
        sharpe = (ann_return - risk_free_rate) / ann_vol
    else:
        sharpe = np.nan

    equity = df['equity']
    roll_max = equity.cummax()
    drawdown = equity / roll_max - 1
    max_dd = drawdown.min()

    return {
        "Cumulative Return": cum_return,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd,
        "Final Equity": float(equity.iloc[-1])  # Convert to float
    }

In [106]:
# Define the ticker symbol (e.g., SPY)
ticker_symbol = 'SPY'

# Define the period for historical data (e.g., '25y' for 25 years)
period = '25y'

# Fetch historical data
stock_data = yf.download(ticker_symbol, period=period, auto_adjust=True)
print(f"\nColumns after yf.download: {stock_data.columns.tolist()}")

# Flatten MultiIndex columns if they exist (yfinance output sometimes has them)
if isinstance(stock_data.columns, pd.MultiIndex):
    # Extract the first level of the MultiIndex as the new column names
    stock_data.columns = [col[0] for col in stock_data.columns.values]
    print(f"Columns after flattening MultiIndex: {stock_data.columns.tolist()}")

# Reset index to make 'Date' a regular column
stock_data.reset_index(inplace=True)
print(f"Columns after reset_index: {stock_data.columns.tolist()}")

# Display the first few rows of the processed data
print(f"Historical data for {ticker_symbol} for the last {period} (processed):")

# Save the DataFrame to a CSV file
output_filename = 'SPY.csv'
# Save processed_stock_data, without writing the DataFrame index to CSV
stock_data.to_csv('sample_data/' + output_filename, index=False)

print(f"Processed historical data saved to {output_filename}")

[*********************100%***********************]  1 of 1 completed


Columns after yf.download: [('Close', 'SPY'), ('High', 'SPY'), ('Low', 'SPY'), ('Open', 'SPY'), ('Volume', 'SPY')]
Columns after flattening MultiIndex: ['Close', 'High', 'Low', 'Open', 'Volume']
Columns after reset_index: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
Historical data for SPY for the last 25y (processed):
Processed historical data saved to SPY.csv


In [107]:
prices = load_price_data('sample_data/SPY.csv')
prices.head()

,Close,High,Low,Open,Volume
Date,,,,,
2000-11-15,88.830772,89.178855,87.677126,88.512525,8837700.0
2000-11-16,87.438477,89.029714,87.398696,88.204259,6684100.0
2000-11-17,86.971031,88.472761,86.404153,87.398676,6551100.0
2000-11-20,85.727882,86.801967,85.489197,86.404158,5458500.0
2000-11-21,86.165451,86.682603,84.981969,85.847204,7684300.0


In [108]:
strategies: Dict[str, Callable[[pd.DataFrame], pd.Series]] = {
    "buy_and_hold_long_only": lambda df: strategy_buy_and_hold(df),
    "sma_long_only_20_50": lambda df: strategy_sma_long_only(
        df, fast_window=20, slow_window=50),
    "sma_long_only_10_30": lambda df: strategy_sma_long_only(
        df, fast_window=10, slow_window=30),
    "sma_long_short_20_50": lambda df: strategy_sma_long_short(
        df, fast_window=20, slow_window=50),
    "sma_long_short_10_30": lambda df: strategy_sma_long_short(
        df, fast_window=10, slow_window=30),
    "mean_reversion_long_short_5_1": lambda df: strategy_mean_reversion_long_short(
        df, lookback=5, z_entry=1.0)
}

for name, strat in strategies.items():
    print(f"\n=== Strategy: {name} ===")
    signal = strat(prices)
    bt = backtest(
        prices,
        signal, # Pass the generated signal to the backtest function
        initial_capital=100_000.0,
        trading_fee_bps=10.0,
    )
    summary = performance_summary(bt)

    for k, v in summary.items():
        print(f"{k:22s}: {v:.4f}")


=== Strategy: buy_and_hold_long_only ===
Cumulative Return     : 6.5634
Annualized Return     : 0.0845
Annualized Volatility : 0.1926
Sharpe Ratio          : 0.4385
Max Drawdown          : -0.5519
Final Equity          : 756340.1528

=== Strategy: sma_long_only_20_50 ===
Cumulative Return     : 2.0145
Annualized Return     : 0.0452
Annualized Volatility : 0.1160
Sharpe Ratio          : 0.3898
Max Drawdown          : -0.2896
Final Equity          : 301450.7801

=== Strategy: sma_long_only_10_30 ===
Cumulative Return     : 2.8330
Annualized Return     : 0.0553
Annualized Volatility : 0.1117
Sharpe Ratio          : 0.4954
Max Drawdown          : -0.2846
Final Equity          : 383303.4184

=== Strategy: sma_long_short_20_50 ===
Cumulative Return     : -0.3388
Annualized Return     : -0.0164
Annualized Volatility : 0.1914
Sharpe Ratio          : -0.0859
Max Drawdown          : -0.5626
Final Equity          : 66119.6126

=== Strategy: sma_long_short_10_30 ===
Cumulative Return     : 0.0162

In [ ]:
strategies: Dict[str, Callable[[pd.DataFrame], pd.Series]] = {
    "buy_and_hold_long_only": lambda df: strategy_buy_and_hold(df),
    "sma_long_only_20_50": lambda df: strategy_sma_long_only(
        df, fast_window=20, slow_window=50),
    "sma_long_only_10_30": lambda df: strategy_sma_long_only(
        df, fast_window=10, slow_window=30),
    "sma_long_short_20_50": lambda df: strategy_sma_long_short(
        df, fast_window=20, slow_window=50),
    "sma_long_short_10_30": lambda df: strategy_sma_long_short(
        df, fast_window=10, slow_window=30),
    "mean_reversion_long_short_5_1": lambda df: strategy_mean_reversion_long_short(
        df, lookback=5, z_entry=1.0)
}

all_summaries = {}
all_equity_curves = pd.DataFrame(index=prices.index)

for name, strat_func in strategies.items():
    print(f"\n=== Strategy: {name} ===")
    signal = strat_func(prices)
    bt = backtest(
        prices,
        signal,
        initial_capital=100_000.0,
        trading_fee_bps=10.0,
    )
    summary = performance_summary(bt)
    all_summaries[name] = summary
    all_equity_curves[name] = bt['equity']

    for k, v in summary.items():
        print(f"{k:22s}: {v:.4f}")

# Convert the dictionary of summaries to a DataFrame for easy comparison
summary_df = pd.DataFrame(all_summaries).T
print("\n--- Consolidated Performance Summary ---")
display(summary_df)

### Strategy Comparison Plots

In [ ]:
# Plotting Equity Curves
plt.figure(figsize=(14, 8))
sns.lineplot(data=all_equity_curves)
plt.title('Strategy Equity Curves')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Strategy')
plt.tight_layout()
plt.show()

In [ ]:
# Plotting Max Drawdown for each strategy (visualizing how low they went)
# First, calculate drawdowns for each strategy in all_equity_curves
all_drawdowns = pd.DataFrame(index=prices.index)
for col in all_equity_curves.columns:
    roll_max = all_equity_curves[col].cummax()
    drawdown = (all_equity_curves[col] / roll_max) - 1
    all_drawdowns[col] = drawdown

plt.figure(figsize=(14, 8))
sns.lineplot(data=all_drawdowns)
plt.title('Strategy Drawdowns')
plt.xlabel('Date')
plt.ylabel('Drawdown (from peak equity)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Strategy')
plt.tight_layout()
plt.show()